# Week 1 Community Contribution - AdnanGobeljic

This notebook combines key ideas from Week 1 into one practical helper:

- Day 1: webpage content extraction for extra context
- Day 2: OpenAI-compatible endpoints (OpenAI + Ollama)
- Day 4: short conversation memory handling
- Week 1 exercise: technical-question tutor

It answers technical questions with both GPT and Llama, then optionally merges both answers into one final explanation.

In [4]:
# imports

import os
import sys
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

# Make sure we can import week1/scraper.py regardless of where this notebook is opened from.
candidate_paths = [
    os.path.abspath("week1"),
    os.path.abspath("."),
    os.path.abspath(".."),
    os.path.abspath(os.path.join("..", "..")),
]
for path in candidate_paths:
    if os.path.isfile(os.path.join(path, "scraper.py")) and path not in sys.path:
        sys.path.insert(0, path)
        break

from scraper import fetch_website_contents


In [ ]:
# setup environment and clients

load_dotenv(override=True)

MODEL_GPT = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MODEL_LLAMA = os.getenv("OLLAMA_MODEL", "llama3.2")

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    print("OPENAI_API_KEY not found. GPT calls will be skipped.")
    openai_client = None
elif openai_api_key.strip() != openai_api_key:
    print("OPENAI_API_KEY appears to have extra spaces. Please clean your .env file.")
    openai_client = None
else:
    openai_client = OpenAI(api_key=openai_api_key)
    print("OpenAI client ready.")

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

try:
    requests.get("http://localhost:11434", timeout=2)
    print("Ollama is running.")
except Exception:
    print("Ollama is not running. In a terminal run: ollama serve")


## Prompt setup

The tutor focuses on practical explanations and concise examples.

In [ ]:
SYSTEM_PROMPT = """
You are a practical technical tutor.
Explain clearly, avoid unnecessary jargon, and include one concrete example.
If the user shares code, explain what it does and why it matters.
Respond in markdown and do not wrap your response in code fences.
"""

def build_user_prompt(question, extra_context=""):
    prompt = f"Technical question:\n{question.strip()}\n\n"
    if extra_context:
        prompt += (
            "Reference context from a webpage is included below. "
            "Use it only if relevant:\n\n"
            f"{extra_context[:3500]}\n\n"
        )
    prompt += (
        "Please respond with these sections:\n"
        "## Quick answer\n"
        "## Step-by-step explanation\n"
        "## Practical example\n"
        "## Common mistakes to avoid"
    )
    return prompt


In [ ]:
def get_website_context(url, max_chars=3500):
    if not url:
        return ""
    try:
        content = fetch_website_contents(url)
        return content[:max_chars]
    except Exception as exc:
        return f"Could not fetch website context from {url}. Error: {exc}"


In [ ]:
def stream_openai_answer(messages, model=MODEL_GPT):
    if openai_client is None:
        print("Skipping OpenAI call (client not configured).")
        return ""

    stream = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )

    answer = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        answer += chunk.choices[0].delta.content or ""
        update_display(Markdown(answer), display_id=display_handle.display_id)
    return answer


def ask_ollama(messages, model=MODEL_LLAMA):
    try:
        response = ollama_client.chat.completions.create(
            model=model,
            messages=messages,
        )
        answer = response.choices[0].message.content
        display(Markdown(answer))
        return answer
    except Exception as exc:
        print(f"Ollama call failed: {exc}")
        return ""


In [ ]:
def technical_tutor(question, context_url=None, conversation_history=None):
    """
    Ask the same question to GPT and Llama, then optionally merge both answers.
    conversation_history keeps a short memory, inspired by Day 4.
    """
    if conversation_history is None:
        conversation_history = []

    context = get_website_context(context_url) if context_url else ""

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(conversation_history[-6:])
    messages.append({"role": "user", "content": build_user_prompt(question, context)})

    print("OpenAI answer (streaming):")
    gpt_answer = stream_openai_answer(messages)

    print("\nOllama answer:")
    llama_answer = ask_ollama(messages)

    merged_answer = ""
    if openai_client is not None and gpt_answer and llama_answer:
        merge_messages = [
            {
                "role": "system",
                "content": (
                    "You are a senior reviewer. Merge two technical answers into one clearer and"
                    " more practical explanation. Keep it concise and accurate."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Answer A:\n{gpt_answer}\n\n"
                    f"Answer B:\n{llama_answer}"
                ),
            },
        ]
        merged = openai_client.chat.completions.create(
            model=MODEL_GPT,
            messages=merge_messages,
        )
        merged_answer = merged.choices[0].message.content
        print("\nMerged answer:")
        display(Markdown(merged_answer))

    final_answer = merged_answer or gpt_answer or llama_answer
    conversation_history.extend([
        {"role": "user", "content": question},
        {"role": "assistant", "content": final_answer},
    ])

    return {
        "gpt_answer": gpt_answer,
        "llama_answer": llama_answer,
        "merged_answer": merged_answer,
        "history": conversation_history,
    }


In [ ]:
# example usage

conversation_history = []

question = """
In Python, what is the difference between a generator expression and a list comprehension?
When should I choose each one in production systems?
"""

context_url = "https://docs.python.org/3/howto/functional.html"

result = technical_tutor(
    question=question,
    context_url=context_url,
    conversation_history=conversation_history,
)


In [ ]:
# follow-up question to demonstrate short memory behavior

follow_up_question = "Can you now show a tiny benchmark approach to compare both choices safely?"

result = technical_tutor(
    question=follow_up_question,
    conversation_history=conversation_history,
)
